# Step 2: Filter notebook

In [1]:
import geopandas as gpd
import pandas as pd
from functools import reduce
import osmnx as ox
from shapely.geometry import Point, box, LineString
import os
import folium
from folium import Choropleth, CircleMarker, GeoJson
import branca.colormap as cm
from IPython.display import display
pd.set_option('display.max_columns', None)

from network_connectivity import *

## Imports

#### Import des segments

In [2]:
operation_crs = "EPSG:2056"  # Swiss coordinate system
target_crs = "EPSG:4326"  # WGS84 coordinate system

input_file_path = '../../Data/input/attributs'
network_file_path = '../../Data/input/network'
output_step1_path='../../Data/output/step-1'
output_step2_path='../../Data/output/step-2'
output_step3_path='../../Data/output/step-3'

save_filtered_attributes = True

# Load segments GeoDataFrame (with 'segment_id')
print("Loading pedestrian segments...")
# reLoad pedestrian segments
segmented_net = gpd.read_parquet(os.path.join(output_step1_path, "step1_pedestrian_segments.parquet"))
segmented_net = segmented_net.to_crs(operation_crs)

# Define a function to save the filtered data
def save(save_filtered_attributes, row, gdf, attribute):
    # Clean geometries first
    gdf["geometry"] = gdf["geometry"].apply(lambda geom: geom.buffer(0) if geom is not None and not geom.is_valid else geom)
    print("Geometries cleaned")

    if not save_filtered_attributes:
        print("Note : Save option is disabled.")
        return

    # Parse save formats: comma-separated list allowed
    raw = str(row.get('save_format', '') or '')
    formats = [f.strip().lower() for f in raw.split(',') if f.strip()]
    if not formats:
        formats = ['csv']  # default fallback

    for fmt in formats:
        try:
            if fmt == 'parquet':
                dirpath = f'{output_step2_path}/parquet_attributs'
                os.makedirs(dirpath, exist_ok=True)
                gdf.to_parquet(f"{dirpath}/{attribute}.parquet")
                print(f"Filtered data saved for attribute: {attribute} in format: parquet")

            elif fmt == 'gpkg' or fmt == 'geopackage':
                dirpath = f'{output_step2_path}/gpkg_attributs'
                os.makedirs(dirpath, exist_ok=True)
                gdf.to_file(f"{dirpath}/{attribute}.gpkg", driver="GPKG")
                print(f"Filtered data saved for attribute: {attribute} in format: gpkg")

            elif fmt == 'csv':
                dirpath = f'{output_step2_path}/csv_attributs'
                os.makedirs(dirpath, exist_ok=True)
                # to_csv may not preserve geometry consistently; keep original behavior
                gdf.to_csv(f"{dirpath}/{attribute}.csv", index=False)
                print(f"Filtered data saved for attribute: {attribute} in format: csv")

            else:
                # Unknown format -> fallback to csv and warn
                dirpath = f'{output_step2_path}/csv_attributs'
                os.makedirs(dirpath, exist_ok=True)
                gdf.to_csv(f"{dirpath}/{attribute}.csv", index=False)
                print(f"Warning: Unknown save format '{fmt}' for attribute {attribute}. Data saved as csv.")
        except Exception as e:
            print(f"Error saving {attribute} as {fmt}: {e}")

Loading pedestrian segments...


#### Import des attributs

In [3]:
attributs_info = pd.read_excel(f"{input_file_path}/attributs_info.xlsx", sheet_name="attributs_info")
attributs_info = attributs_info[attributs_info['include_in_index'] != False]

**Connectivité du réseau**

In [4]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'connectivite'
###--------------------
row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
print(f"Processing attribute: {attribute}")

# Ensure geometry column is set to avoid spatial index errors
segmented_net = segmented_net.set_geometry("geometry")
print(segmented_net.columns)

# 1. Ajouter u, v, key si besoin
segmented_net = add_uv_columns(segmented_net)

# 2. Calculer les métriques et le score d’alternatives
segmented_net_index = compute_connectivity_metrics(
    segmented_net,
    buffer_m=35,
    compute_betweenness=False,
    betweenness_k=None,
    crs_meter_epsg=2056,
    main_metrics_only=False,
    conn_index_metric="conn_branching_in_buffer",
)

segmented_net_index['filtered'] = 1


# Preview
segmented_net_index.to_crs(target_crs).head()

Data initialized
Processing attribute: connectivite
Index(['Largeur', 'Partage_us', 'Objet', 'Revetement', 'Vitesse', 'Zone_mod',
       'Commune', 'Type', 'Pente', 'Classe', 'Nom_voie', 'Franchisse',
       'PP_Feux', 'SHAPE_Leng', 'geometry', 'length_m', 'length',
       'segment_id'],
      dtype='object')


,Largeur,Partage_us,Objet,Revetement,Vitesse,Zone_mod,Commune,Type,Pente,Classe,Nom_voie,Franchisse,PP_Feux,SHAPE_Leng,geometry,length_m,length,segment_id,u,v,key,conn_deadend_flag,conn_intersection_flag,conn_nodes_in_buffer,conn_intersections_in_buffer,conn_branching_in_buffer,conn_index_score,filtered
0,Moyen,Mixité vélos - Piste sur trotto,Trottoir,Béton bitumineux,50,None,Onex,Trottoir,0.6,tertiaire,Route de Chancy,None,Non,2.681935,"LINESTRING (6.09574 46.18116, 6.09576 46.18114)",2.681935,2.681935,000000,5415881993839729181,-8869193047698984299,000000,True,False,13,1,0.000000,0.109677,1
1,Etroit,Mixité vélos - Piste sur trotto,Trottoir,Béton bitumineux,50,None,Veyrier,Trottoir,3.7,tertiaire,Chemin de Pinchat,None,Oui,6.862863,"LINESTRING (6.14937 46.17581, 6.14939 46.17584...",6.862863,6.862863,000001,4193581936861870467,-7476120331322857847,000001,True,False,10,0,0.000000,0.100000,1
2,Moyen,Mixité vélos - Piste sur trotto,Trottoir,Béton bitumineux,50,None,Genève-Eaux-Vives,Trottoir,0.0,tertiaire,None,None,None,42.163226,"LINESTRING (6.16722 46.20953, 6.16721 46.20956...",42.163226,27.085935,000002,29981410322409309,7420733444885395478,000002,False,True,31,13,0.312903,0.312903,1
3,Moyen,Mixité vélos - Piste sur trotto,Trottoir,Béton bitumineux,50,None,Genève-Eaux-Vives,Trottoir,0.0,tertiaire,None,None,None,42.163226,"LINESTRING (6.16732 46.20941, 6.16729 46.20945...",42.163226,15.077290,000003,5038634793733427015,29981410322409309,000003,False,True,48,17,0.409677,0.409677,1
4,Large,Mixité vélos - Piste sur trotto,Trottoir,Béton bitumineux,50,None,Vernier,Trottoir,0.7,tertiaire,Avenue du Lignon,None,Non,15.240382,"LINESTRING (6.09883 46.20446, 6.09883 46.20445...",15.240382,4.338167,000004,-3033272787963850611,7254579389179832639,000004,False,True,43,10,0.235484,0.235484,1


In [5]:
save(save_filtered_attributes, row, segmented_net_index.to_crs(target_crs)[["segment_id", "Objet","geometry","conn_index_score","filtered"]], attribute)

Geometries cleaned
Filtered data saved for attribute: connectivite in format: parquet
Filtered data saved for attribute: connectivite in format: csv
Filtered data saved for attribute: connectivite in format: gpkg


In [6]:
segmented_net_index.to_crs(target_crs)[["key","Objet","geometry","conn_index_score", "filtered"]]

,key,Objet,geometry,conn_index_score,filtered
0,000000,Trottoir,"LINESTRING (6.09574 46.18116, 6.09576 46.18114)",0.109677,1
1,000001,Trottoir,"LINESTRING (6.14937 46.17581, 6.14939 46.17584...",0.100000,1
2,000002,Trottoir,"LINESTRING (6.16722 46.20953, 6.16721 46.20956...",0.312903,1
3,000003,Trottoir,"LINESTRING (6.16732 46.20941, 6.16729 46.20945...",0.409677,1
4,000004,Trottoir,"LINESTRING (6.09883 46.20446, 6.09883 46.20445...",0.235484,1
...,...,...,...,...,...
287269,287269,Chaussée,"LINESTRING (6.15824 46.27616, 6.15829 46.27617...",0.100000,1
287270,287270,Chemin,"LINESTRING (6.09656 46.15491, 6.09663 46.15494...",0.109677,1
287271,287271,Chemin,"LINESTRING (6.09669 46.15496, 6.09671 46.15497)",0.109677,1
287272,287272,Chemin,"LINESTRING (6.09639 46.15483, 6.09643 46.15485...",0.109677,1
